# Model Controller Tutorial: EnviBert model (MultiHead)

> This notebook contains some example of how to use the EnviBert-based models in this NLP library

We will walk through other cases of classification: multi-head and multi-label. Since we will showcase the capabiilty of this label in these cases, there won't be as detailed as [this tutorial](https://anhquan0412.github.io/that-nlp-library/model_main_envibert.html)

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
#| hide
from nbdev.showdoc import *

## Load data

In [ ]:
from that_nlp_library.text_transformation import *
from that_nlp_library.text_augmentation import *
from that_nlp_library.text_main import *

In [ ]:
from underthesea import text_normalize
from functools import partial
from pathlib import Path
from importlib.machinery import SourceFileLoader
import os
from transformers import DataCollatorWithPadding


Define some necessary text augmentations and text transformations

> For Text Transformation

In [ ]:
txt_tfms=[text_normalize]

> For Text Augmentation

In [ ]:
over_nonown_tfm = partial(sampling_with_condition,query='Source=="non owned"',frac=0.5,seed=42,apply_to_all=False)
over_nonown_tfm.__name__ = 'Oversampling Non Owned'

over_own_tfm = partial(sampling_with_condition,query='Source=="owned"',frac=2,seed=42,apply_to_all=False)
over_own_tfm.__name__ = 'Oversampling Owned'

over_hc_tfm = partial(sampling_with_condition,query='Source=="hc search"',frac=2.5,seed=42,apply_to_all=False)
over_hc_tfm.__name__ = 'Oversampling HC search'

remove_accent_tfm = partial(remove_vnmese_accent,frac=1,seed=42,apply_to_all=True)
remove_accent_tfm.__name__ = 'Add No-Accent Text'

aug_tfms = [over_nonown_tfm,over_own_tfm,over_hc_tfm,remove_accent_tfm]

Create a TextDataMain object

In [ ]:
DATA_PATH = Path('secret_data')

In [ ]:
tdm = TextDataMain.from_csv(DATA_PATH/'buyer_listening_with_all_raw_data_w151617.csv',
                            return_df=False,
                            main_content='Content',
                            metadatas='Source',
                            label_names=['L1','L2'],
                            val_ratio=0.24,
                            split_cols='L2',
                            content_tfms = txt_tfms,
                            aug_tfms = aug_tfms,
                            process_metadatas=True,
                            seed=42,
                            shuffle_trn=True)

----- Input Validation Precheck -----
DataFrame contains missing values!
-----> List of columns and the number of missing values for each
is_valid    65804
dtype: int64
DataFrame contains duplicated values!
-----> Number of duplications: 7 rows


Define our tokenizer for EnviBert

In [ ]:
cache_dir=Path('./envibert_tokenizer')
tokenizer = SourceFileLoader("envibert.tokenizer", 
                             str(cache_dir/'envibert_tokenizer.py')).load_module().RobertaTokenizer(cache_dir)

EnviBert a data collator to work. We will save this as an attribute in TDM

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer,padding=True,max_length=512)
tdm.set_data_collator(data_collator)

Create our DatasetDict from TextDataMain (as our `ModelController` class can also work with DatasetDict)

In [ ]:
main_ddict= tdm.to_datasetdict(tokenizer,
                               max_length=512,
                               trn_ratio=0.1)

-------------------- Start Main Text Processing --------------------
----- Metadata Simple Processing & Concatenating to Main Content -----
----- Label Encoding -----
-------------------- Text Transformation --------------------
----- text_normalize -----


100%|██████████████████████████████| 112453/112453 [00:28<00:00, 3887.59it/s]


-------------------- Train Test Split --------------------
Previous Validation Percentage: 24.0%
- Before leak check
Size: 26989
- After leak check
Size: 23928
- Number of rows leaked: 3061, or 11.34% of the original validation (or test) data
Current Validation Percentage: 21.278%
-------------------- Text Augmentation --------------------
Train data size before augmentation: 88525
----- Oversampling Non Owned -----
Train data size after THIS augmentation: 98333
----- Oversampling Owned -----
Train data size after THIS augmentation: 109165
----- Oversampling HC search -----
Train data size after THIS augmentation: 116057
----- Add No-Accent Text -----


100%|█████████████████████████████| 116057/116057 [00:06<00:00, 18823.29it/s]


Train data size after THIS augmentation: 232114
Train data size after ALL augmentation: 232114
-------------------- Map Tokenize Function --------------------


Map:   0%|          | 0/23211 [00:00<?, ? examples/s]

Map:   0%|          | 0/23928 [00:00<?, ? examples/s]

In [ ]:
main_ddict

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'Source', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 23211
    })
    validation: Dataset({
        features: ['text', 'label', 'Source', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 23928
    })
})

# Model Experiment: EnviBert Multi-Head Classification (with Hidden Layer Concatenation)

In [ ]:
from that_nlp_library.models.classifiers import *

In [ ]:
from that_nlp_library.model_main import *

comet_ml is installed but `COMET_API_KEY` is not set.


In [ ]:
from sklearn.metrics import f1_score, accuracy_score

## Train EnviBert (with hidden layer concatenation), using TDM

Let's create our model controller

In [ ]:
model_name='nguyenvulebinh/envibert'
num_classes = [len(tdm.label_lists[0]),len(tdm.label_lists[1])] 

_model_kwargs={
    'concathead_class': RobertaConcatHeadSimple,
    'classifier_dropout':0.1,
    'last_hidden_size':768,  
    'is_multilabel':tdm.is_multilabel, 
    'is_multihead':tdm.is_multihead, # change to True
    'head_class_sizes': num_classes,
    'head_weights':[1,2], # weights for L1 and L2
}

model = model_init_classification(model_class = RobertaHiddenStateConcatForSequenceClassification,
                                  cpoint_path = 'nguyenvulebinh/envibert', 
                                  output_hidden_states=True, # since we are using 'hidden layer contatenation'
                                  seed=42,
                                  model_kwargs = _model_kwargs)
metric_funcs = [partial(f1_score,average='macro'),accuracy_score]
controller = ModelController(model,tdm,metric_funcs)

Some weights of the model checkpoint at nguyenvulebinh/envibert were not used when initializing RobertaHiddenStateConcatForSequenceClassification: ['lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.bias', 'lm_head.layer_norm.weight', 'lm_head.layer_norm.bias']
- This IS expected if you are initializing RobertaHiddenStateConcatForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaHiddenStateConcatForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaHiddenStateConcatForSequenceClassification were not initialized from the model checkpoint at nguyenvulebinh/envibert and are newly initialized: ['classification_head.out_proj.bia

And we can start training our model

In [ ]:
lr = 8.2e-5
bs=8
wd=0.01
epochs= 4

In [ ]:
controller.fit(epochs,lr,
               batch_size=bs,
               weight_decay=wd,
               save_checkpoint=False,
#                o_dir='sample_weights',
               compute_metrics=compute_metrics_multihead_classification,
              )

# Epoch	Training Loss	Validation Loss	F1 Score L1	Accuracy Score L1
# 1	No log	0.770289	0.633258	0.749269
# 2	0.857600	0.710960	0.689167	0.770079
# 3	0.857600	0.746624	0.698602	0.775512
# 4	0.354300	0.811047	0.700496	0.776139

# Equal weights
# Epoch	Training Loss	Validation Loss	F1 Score L1	Accuracy Score L1	F1 Score L2	Accuracy Score L2
# 1	No log	2.209617	0.622977	0.741307	0.196366	0.627549
# 2	2.476700	1.915091	0.692587	0.765965	0.281379	0.669843
# 3	2.476700	1.854167	0.696627	0.776204	0.328412	0.689694
# 4	1.101100	1.894282	0.699866	0.777666	0.330808	0.692578

# L1 1 L2 2
# Epoch	Training Loss	Validation Loss	F1 Score L1	Accuracy Score L1	F1 Score L2	Accuracy Score L2
# 1	No log	3.735447	0.614677	0.723587	0.196626	0.615388
# 2	4.016400	3.096701	0.683411	0.762496	0.304085	0.669425
# 3	4.016400	2.957187	0.698510	0.777583	0.341109	0.694040
# 4	1.739400	3.008255	0.700440	0.775242	0.349905	0.695127

/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2372: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Epoch,Training Loss,Validation Loss,F1 Score L1,Accuracy Score L1,F1 Score L2,Accuracy Score L2
1,No log,3.735447,0.614677,0.723587,0.196626,0.615388
2,4.016400,3.096701,0.683411,0.762496,0.304085,0.669425
3,4.016400,2.957187,0.698510,0.777583,0.341109,0.694040
4,1.739400,3.008255,0.700440,0.775242,0.349905,0.695127


/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2372: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2372: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2372: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [ ]:
controller.trainer.model.save_pretrained('./sample_weights/my_model')

## Predict using trained model, using TDM

### Load trained model

In [ ]:
model = model_init_classification(model_class = RobertaHiddenStateConcatForSequenceClassification,
                                  cpoint_path = 'sample_weights/my_model', 
                                  output_hidden_states=True,
                                  seed=42,
                                  model_kwargs = _model_kwargs)
metric_funcs = [partial(f1_score,average='macro'),accuracy_score]
controller = ModelController(model,tdm,metric_funcs)

### Predict Train/Validation set

Make prediction on all validation set

In [ ]:
df_val = controller.predict_ddict(ds_type='validation')

-------------------- Start making predictions --------------------


Map:   0%|          | 0/23928 [00:00<?, ? examples/s]

/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2372: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [ ]:
df_val.head()

,text,label,Source,pred_L1,pred_prob_L1,pred_L2,pred_prob_L2
0,google play - Chuyên hủy đơn vô lí do trong kh...,"[8, 17]",google play,Order/Item,0.918496,Order cancelled,0.932477
1,non owned - Mn ơi fl chéo với shop em ạ Trả ng...,"[5, 52]",non owned,Others,0.997265,Seller,0.955981
2,non owned - Ai dùng ngân hàng VCB chưa lk shop...,"[5, 12]",non owned,Others,0.882347,Cannot defined,0.773110
3,hc search - áo size s,"[5, 12]",hc search,Others,0.998028,Cannot defined,0.998320
4,google play - Laggghh,"[3, 5]",google play,Feature,0.974982,App performance,0.962226


To convert the label index to string, we can use the ```label_lists``` attribute of tdm

In [ ]:
import pandas as pd

In [ ]:
df_val[['label_L1','label_L2']] = pd.DataFrame(df_val.label.tolist(), index= df_val.index)

In [ ]:
df_val.head()

,text,label,Source,pred_L1,pred_prob_L1,pred_L2,pred_prob_L2,label_L1,label_L2
0,google play - Chuyên hủy đơn vô lí do trong kh...,"[8, 17]",google play,Order/Item,0.918496,Order cancelled,0.932477,8,17
1,non owned - Mn ơi fl chéo với shop em ạ Trả ng...,"[5, 52]",non owned,Others,0.997265,Seller,0.955981,5,52
2,non owned - Ai dùng ngân hàng VCB chưa lk shop...,"[5, 12]",non owned,Others,0.882347,Cannot defined,0.773110,5,12
3,hc search - áo size s,"[5, 12]",hc search,Others,0.998028,Cannot defined,0.998320,5,12
4,google play - Laggghh,"[3, 5]",google play,Feature,0.974982,App performance,0.962226,3,5


In [ ]:
df_val['label_L1']= df_val['label_L1'].apply(lambda x: tdm.label_lists[0][x]).values
df_val['label_L2']= df_val['label_L2'].apply(lambda x: tdm.label_lists[1][x]).values

In [ ]:
f1_score(df_val.label_L1,df_val.pred_L1,average='macro'),f1_score(df_val.label_L2,df_val.pred_L2,average='macro')

(0.7005934974488988, 0.3498784748226089)

### Predict Test set

We will go through details on how to make a prediction on a completely new and raw dataset using our trained model. For now, let's reuse the sample csv and pretend it's our test set

In [ ]:
df_test = TextDataMain.from_csv(Path('sample_data')/'sample.csv',return_df=True)

----- Input Validation Precheck -----


We will remove all the labels and unnecessary columns

In [ ]:
df_test = df_test.drop(['L1','L2'],axis=1)

In [ ]:
df_test.head()

,Group,Source,Content
0,Google Play,Google Play,Mình khuyên các bạn nên mua bên Lazada hoặc Ti...
1,Google Play,Google Play,Con cc quoảng cáu ít thôi
2,iOS,iOS,Mình có một vài món hàng shipper ấn giao r mà ...
3,Google Play,Google Play,Mình đã sử dụng shoppe cũng 1 thời gian dài rồ...
4,Google Play,Google Play,Chăm sóc khách hàng quá tệ. Nhân viên hỗ trợ c...


We will create a DatasetDict for this test dataframe

In [ ]:
test_ddict = tdm.get_test_datasetdict_from_df(df_test)

-------------------- Getting Test Set --------------------
----- Input Validation Precheck -----
-------------------- Start Test Set Transformation --------------------
----- Metadata Simple Processing & Concatenating to Main Content -----
-------------------- Text Transformation --------------------
----- text_normalize -----


100%|██████████████████████████████████████| 70/70 [00:00<00:00, 6164.21it/s]

-------------------- Test Leak Checking --------------------
- Before leak check
Size: 70


- After leak check
Size: 0
- Number of rows leaked: 70, or 100.00% of the original validation (or test) data
-------------------- Construct DatasetDict --------------------


Map:   0%|          | 0/70 [00:00<?, ? examples/s]

Remember the ***Leak Check*** we did in TextDataMain? Our ```df_test``` only has 70 rows, and it also shows that 70 rows of our data is leaked (100%), which is correct because this test dataset is actually a small sample of the training data.

In [ ]:
test_ddict

DatasetDict({
    test: Dataset({
        features: ['text', 'Source', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 70
    })
})

Our test data has been processed + transformed (but not augmented) the same way as the validation set. Now we can start making the prediction

In [ ]:
controller = ModelController(model,tdm)
df_result = controller.predict_ddict(ddict=test_ddict,ds_type='test')

-------------------- Start making predictions --------------------


Map:   0%|          | 0/70 [00:00<?, ? examples/s]

/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2372: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [ ]:
df_result.head()

,text,Source,pred_L1,pred_prob_L1,pred_L2,pred_prob_L2
0,google play - Mình khuyên các bạn nên mua bên ...,google play,Services,0.344306,Contact Agent,0.304143
1,google play - Con cc quoảng cáu ít thôi,google play,Others,0.976592,Cannot defined,0.985818
2,ios - Mình có một vài món hàng shipper ấn giao...,ios,Delivery,0.993229,Driver,0.470777
3,google play - Mình đã sử dụng shoppe cũng 1 th...,google play,Feature,0.490832,Contact Agent,0.788076
4,google play - Chăm sóc khách hàng quá tệ . Nhâ...,google play,Services,0.974205,Contact Agent,0.980829


We can even predict top k results

In [ ]:
df_result = controller.predict_ddict(ddict=test_ddict,ds_type='test',topk=3)
df_result.head()

-------------------- Start making predictions --------------------


Map:   0%|          | 0/70 [00:00<?, ? examples/s]

/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2372: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


,text,Source,pred_L1,pred_prob_L1,pred_L2,pred_prob_L2,pred_L1_top1,pred_L1_top2,pred_L1_top3,pred_prob_L1_top1,pred_prob_L1_top2,pred_prob_L1_top3,pred_L2_top1,pred_L2_top2,pred_L2_top3,pred_prob_L2_top1,pred_prob_L2_top2,pred_prob_L2_top3
0,google play - Mình khuyên các bạn nên mua bên ...,google play,"[8, 0, 7]","[0.3443061, 0.3106306, 0.23096871]","[17, 20, 25]","[0.30414346, 0.1986542, 0.18507561]",Services,Buyer complained seller,Return/Refund,0.344306,0.310631,0.230969,Contact Agent,Customer service (didn't respond/impolite),Dispute,0.304143,0.198654,0.185076
1,google play - Con cc quoảng cáu ít thôi,google play,"[5, 1, 3]","[0.9765921, 0.01907897, 0.0015083755]","[12, 33, 44]","[0.98581773, 0.004449921, 0.004408383]",Others,Commercial,Feature,0.976592,0.019079,0.001508,Cannot defined,Items/price,Promotions,0.985818,0.004450,0.004408
2,ios - Mình có một vài món hàng shipper ấn giao...,ios,"[2, 4, 3]","[0.9932294, 0.003656316, 0.0012397956]","[26, 23, 22]","[0.4707766, 0.3193537, 0.15232192]",Delivery,Order/Item,Feature,0.993229,0.003656,0.001240,Driver,Delivery time,Delivery status/info,0.470777,0.319354,0.152322
3,google play - Mình đã sử dụng shoppe cũng 1 th...,google play,"[3, 8, 5]","[0.4908319, 0.34037182, 0.059976608]","[17, 5, 12]","[0.78807616, 0.06821524, 0.018510392]",Feature,Services,Others,0.490832,0.340372,0.059977,Contact Agent,App performance,Cannot defined,0.788076,0.068215,0.018510
4,google play - Chăm sóc khách hàng quá tệ . Nhâ...,google play,"[8, 3, 5]","[0.9742053, 0.008464353, 0.006848225]","[17, 59, 12]","[0.9808292, 0.0029095071, 0.0019473273]",Services,Feature,Others,0.974205,0.008464,0.006848,Contact Agent,Shopee Policies,Cannot defined,0.980829,0.002910,0.001947


If we just want to make a prediction on a small amount of data (single sentence, or a few sentences), we can use `ModelController.predict_raw_text`

In [ ]:
# Since we have some metadatas, we need to define a dictionary (to imitate a DatasetDict)
raw_content={
    'Source': 'Google play',
    'Content':'Tôi không thích Shopee.Tại vì dùng app rất chậm,lag banh nhà lầu, thậm chí log in còn không đc'
}

If we don't use metadata, we can use something like this: 

```raw_content='Tôi không thích Shopee.Tại vì dùng app rất chậm,lag banh nhà lầu, thậm chí log in còn không đc'```

In [ ]:
df_result = controller.predict_raw_text(raw_content,topk=1)
df_result

100%|████████████████████████████████████████| 1/1 [00:00<00:00, 4471.54it/s]


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2372: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


,text,Source,pred_L1,pred_prob_L1,pred_L2,pred_prob_L2
0,google play - Tôi không thích Shopee . Tại vì ...,google play,Feature,0.995096,App performance,0.991692


In [ ]:
raw_content={
    'Source': ['Google play','Owned'],
    'Content':['Tôi không thích Shopee.Tại vì dùng app rất chậm,lag banh nhà lầu, thậm chí log in còn không đc','App này xài được']
            }
df_result = controller.predict_raw_text(raw_content,topk=2)
df_result

100%|████████████████████████████████████████| 2/2 [00:00<00:00, 7605.27it/s]


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2372: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


,text,Source,pred_L1,pred_prob_L1,pred_L2,pred_prob_L2,pred_L1_top1,pred_L1_top2,pred_prob_L1_top1,pred_prob_L1_top2,pred_L2_top1,pred_L2_top2,pred_prob_L2_top1,pred_prob_L2_top2
0,google play - Tôi không thích Shopee . Tại vì ...,google play,"[3, 9]","[0.9950956, 0.002345441]","[5, 65]","[0.9916922, 0.0016225452]",Feature,Shopee account,0.995096,0.002345,App performance,Sign up/Log in,0.991692,0.001623
1,owned - App này xài được,owned,"[3, 5]","[0.83267194, 0.1540744]","[5, 12]","[0.8209603, 0.14227916]",Feature,Others,0.832672,0.154074,App performance,Cannot defined,0.820960,0.142279
